# 05 — High-NA lithography through the propagation pipeline

A dense mask is imaged into a dielectric of index **1.46**, with geometrical **NA ≈ 1.10**. Every interface calculation and outgoing reconstruction uses the public `PropagationPipeline`. The geometry, mask and incident power (1 W) are fixed.

The main comparison is between images formed by the field:

- **Reference, $t_s=t_p=1$:** transverse intensity $I_\perp=\langle |E_x|^2+|E_y|^2\rangle$.
- **Fresnel:** total electric intensity $I_E=\langle |E_x|^2+|E_y|^2+|E_z|^2\rangle$, for the same incident field.
- **Prepared illumination:** total electric intensity $I_E$, with physical Fresnel coefficients and incident polarization prepared from x instead of y.

The brackets average the two mutually incoherent illumination directions. They do **not** average different polarization preparations. The last case tests whether changing the illumination polarization restores image detail. Polarization ellipses appear separately as diagnostics; the main images and resolution profiles show intensity.

The transverse unity reference has the scalar fringe shape for the symmetric two-order model discussed below. For the full finite mask it remains the transverse intensity of the computed unity-transmission field. Electric intensity is reported in relative units; transmitted power is calculated separately from the Poynting flux.

The pipeline receives an `OpticalSystem`. This benchmark explicitly uses `SpectralFresnel` for its Fresnel/unity-amplitude controls; E/H is requested for power observables. Stored numerical results precede the API refactor; their spectral calculation is unchanged.


In [ ]:
from dataclasses import replace
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import vecdiff_general as optics
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from vecdiff_general.fourier.cartesian import IFFT2, frequency_grid
from vecdiff_general.fourier.nufft import NUFFT
from vecdiff_general.visualization import plot_polarization

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "image.aspect": "equal"})


# --- Wavelength and units ----------------------------------------------------
wavelength = 532e-9  # m
um = 1e-6  # plotting unit
k0 = 2 * np.pi / wavelength


# --- Media: air before the interface, dielectric after it ---------------------
index = 1.46
air = optics.HomogeneousRegion("air", optics.Dielectric(1))
image_medium = optics.HomogeneousRegion("image medium", optics.Dielectric(index))


# --- The three compared cases: (illumination polarization, coefficients) -------
cases = {
    "Reference": ("TM", "unity"),
    "Fresnel": ("TM", "fresnel"),
    "Prepared illumination": ("TE", "fresnel"),
}

# Plot labels name the transmission and the intensity observable of each case
labels = {
    "Reference": r"Reference: $t_s=t_p=1$, $I_\perp$",
    "Fresnel": r"Fresnel: $I_E$",
    "Prepared illumination": r"Prepared illumination: $I_E$",
}

## Optical geometry

The spherical central region has radius $R=30\lambda_0$. Its on-axis virtual-object and real-image conjugates are $B=R(1+n)$ and $F=R(1+1/n)$, with nominal transverse magnification $m=1/n^2$. Aplanatism describes these conjugates of the optical system. The finite mask extends off axis, so distortion and aberration remain possible.

The actual interface is a single-valued **aspheric graph**: the sphere ends at $r_j=0.985R$, then its slope decreases linearly to zero over $0.12R$. Height and slope are continuous; the exterior is planar. This specified continuation makes the whole computational chart valid. It is part of the physical model, not a clipped sphere or a finite lens rim. The quoted core NA is $n r_j/\sqrt{r_j^2+(F-h(r_j))^2}$; outgoing spectral power is measured below to check that high angles are actually populated.

Illumination is specified by propagating a virtual-mask spectrum backwards from $z=B$ to the entrance plane. This prescribes a converging incident field; the upstream optical system that would prepare it is outside this example. All library lengths are in metres.

In [ ]:
# --- Spherical core and its aplanatic conjugates ------------------------------
radius = 30 * wavelength
object_z = radius * (1 + index)  # virtual mask plane, B
image_z = radius * (1 + 1 / index)  # image plane, F
magnification = 1 / index**2


# --- Where the sphere ends and how the continuation flattens -------------------
join = 0.985 * radius
transition = 0.12 * radius

join_height = radius - np.sqrt(radius**2 - join**2)
join_slope = join / np.sqrt(radius**2 - join**2)


# --- Surface height and slope as functions of the radial coordinate -----------
def sag(r):
    """Sphere for r ≤ join; beyond it the slope decreases linearly to zero."""
    q = np.minimum(r, join)
    t = np.clip(r - join, 0, transition)

    sphere = q**2 / (radius + np.sqrt(radius**2 - q**2))
    continuation = join_height + join_slope * (t - t**2 / (2 * transition))

    return np.where(r <= join, sphere, continuation)


def slope(r):
    """Derivative of sag: continuous at the join and zero beyond the transition."""
    q = np.minimum(r, join)
    t = np.clip(r - join, 0, transition)

    sphere = q / np.sqrt(radius**2 - q**2)
    continuation = join_slope * (1 - t / transition)

    return np.where(r <= join, sphere, continuation)


# --- The optical interface ----------------------------------------------------
surface = optics.AxisymmetricSurface(sag, slope)
interface = optics.DielectricInterface(surface, air, image_medium)


# --- Geometrical NA of the spherical core ------------------------------------
core_angle = np.arctan2(join, image_z - join_height)
core_na = index * np.sin(core_angle)

print(f"Core NA: {core_na:.3f}; half-angle: {np.degrees(core_angle):.2f}°")

The meridional profile with the virtual mask plane, the image plane and the rays from the core edge:

In [ ]:
# --- Meridional coordinate, beyond the transition on both sides ---------------
meridian = np.linspace(-1.3, 1.3, 501) * radius


fig, ax = plt.subplots(figsize=(6, 4.2), layout="constrained")

# Interface profile, drawn with z horizontal
ax.plot(sag(abs(meridian)) / um, meridian / um, label="Specified interface")

# Image and virtual-mask planes
ax.axvline(image_z / um, color="C3", label="Image plane")
ax.axvline(object_z / um, color="C2", ls="--", label="Virtual mask plane")

# Marginal rays from the edge of the spherical core to the focus
for sign in (-1, 1):
    ax.plot([join_height / um, image_z / um], [sign * join / um, 0], "k:", lw=0.8)

ax.set(
    xlabel="z (µm)",
    ylabel="y (µm)",
    aspect="equal",
    title="High-NA immersion geometry",
)
ax.legend()
plt.show()

## Dense mask and illumination

The mask contains combs, a connected meander and a contact array. Nominal image track and gap widths in $y$ are $\lambda_0/(2\sqrt{2}n)$, approximately **129 nm**. Contacts have a larger x pitch; their x resolution is measured separately from fine y resolution.

**A–A** marks the longitudinal section at fixed image x. It crosses the lower-left comb and upper-left meander. **B–B** identifies the contact-separation profile in the lower-right array. Both are drawn on the focal images below.

In [ ]:
# --- Lattice spacing in the virtual-mask plane --------------------------------
step_y = index * wavelength / (2 * np.sqrt(2))
step_x = 1.6 * wavelength


# --- Mask layout: 17 rows × 11 columns; True cells are openings ----------------
mask = np.zeros((17, 11), dtype=bool)

# Lower-left comb: four tracks joined by a spine in column 0
mask[0:7:2, 0:4] = True
mask[0:7, 0] = True

# Upper-left meander: four tracks joined alternately at either end
mask[10:17:2, 0:4] = True
for j, row in enumerate(range(10, 16, 2)):
    mask[row : row + 3, 0 if j % 2 else 3] = True

# Lower-right contact array
mask[0:7:2, 7:11:2] = True

# Upper-right comb: four tracks joined by a spine in column 10
mask[10:17:2, 7:11] = True
mask[10:17, 10] = True


# --- Opening centers in the virtual-mask plane (metres) ------------------------
rows, columns = np.nonzero(mask)
centers = np.column_stack(((columns - 5) * step_x, (rows - 8) * step_y))


# --- The same lattice at nominal image scale ----------------------------------
pitch_x = magnification * step_x
pitch_y = magnification * step_y

print(f"Nominal image y track/gap: {pitch_y * 1e9:.1f} nm")
print(f"Nominal image x contact pitch: {2 * pitch_x * 1e9:.1f} nm")


# --- Measurement positions in the image plane ----------------------------------
cut_x = -3 * pitch_x  # section A–A
contact_y = -6 * pitch_y  # profile B–B

# Close-up box at the right ends of the lower-left comb tracks
edge_xlimits = np.array([-2.8, -0.75]) * pitch_x
edge_ylimits = np.array([-9, -1.2]) * pitch_y

# Row edges of the mask, used to draw the mask slice beside section A–A
y_edges = (np.arange(18) - 8.5) * pitch_y / um

### Illumination

Two mutually incoherent illuminations have transverse carriers $k_y=\pm\pi/(2s_y)$, where the virtual-mask row spacing is $s_y=n\lambda_0/(2\sqrt2)$. Each illumination remains coherent across the mask. The finite rectangular cells have an exact Fourier transform: the NUFFT sums their centers and sinc factors supply their widths. A common angular window is unity through incident $\sin\theta=0.49$, tapering to zero at 0.52.

For each mode, the projected x or y polarization vector is normalized to unit length. Consequently the TE-like and TM-like sources have identical angular power, not merely identical transverse electric components. A single flux normalization gives the two-source ensemble 1 W. Incoherent fields are combined as intensities and Stokes parameters, never as complex amplitudes.

In [ ]:
def angular_window(sine):
    """Unity through sin θ = 0.49, tapering to zero at 0.52."""
    taper = np.clip((sine - 0.49) / 0.03, 0, 1)
    window = np.cos(np.pi / 2 * taper) ** 4

    window[sine >= 0.52] = 0
    return window


def opening_spectrum(q):
    """Exact Fourier transform of the rectangular mask openings."""
    form = np.sinc(q[..., 0] * step_x / (2 * np.pi)) * np.sinc(
        q[..., 1] * step_y / (2 * np.pi)
    )
    weights = np.full(len(centers), step_x * step_y)
    sums = NUFFT(weights, centers, q, sign=-1, tolerance=1e-11)
    return sums * form


def prepare_sources(count=384, width=120 * wavelength):
    """TM-like and TE-like sources at z = 0, each two incoherent carriers, 1 W total."""

    # --- Virtual-mask plane z = B and its spectral grid --------------------------
    axis = (np.arange(count) - count // 2) * width / count
    virtual_grid = optics.CartesianGrid(axis, axis, optics.Frame([0, 0, object_z]))

    fx, fy = frequency_grid(virtual_grid.coordinates)
    kxy = 2 * np.pi * np.stack((fx, fy), axis=-1)
    sine = np.linalg.norm(kxy, axis=-1) / k0

    # --- Common angular window and propagation directions ------------------------
    window = angular_window(sine)

    kz = np.sqrt(np.maximum(1 - sine**2, 0))
    direction = np.concatenate((kxy / k0, kz[..., None]), axis=-1)
    direction[sine >= 0.52] = [0, 0, 1]  # outside the window; never used

    # --- Each carrier ±k_y: polarize, synthesize at z = B, propagate to z = 0 ----
    result = {"TM": [], "TE": []}

    for sign in (-1, 1):
        carrier = [0, sign * np.pi / (2 * step_y)]
        q = kxy - carrier
        amplitude = opening_spectrum(q) * window / width**2

        for polarization, component in (("TM", 1), ("TE", 0)):
            # Project y (TM-like) or x (TE-like) onto each mode; unit length
            vector = np.eye(3)[component] - direction * direction[..., component, None]
            vector /= np.linalg.norm(vector, axis=-1)[..., None]

            # Transverse field at the virtual-mask plane
            values = count**2 * IFFT2(amplitude[..., None] * vector[..., :2])
            virtual_field = optics.TransverseElectricField(
                virtual_grid, wavelength, values
            )

            # Plane-wave spectrum in air, moved back to the entrance plane z = 0
            virtual = optics.angular_spectrum(virtual_field, air, evanescent="discard")
            entrance = optics.PlaneTransport().spectrum(virtual, -object_z)
            result[polarization].append(entrance)

    # --- One flux normalization: the two-source ensemble carries 1 W ------------
    power = sum(source.propagate(0).power() / 2 for source in result["TM"])

    for polarization in result:
        result[polarization] = [
            optics.PlaneSpectrum(
                source.grid,
                replace(
                    source.expansion,
                    amplitudes=source.expansion.amplitudes / np.sqrt(power),
                ),
            )
            for source in result[polarization]
        ]

        ensemble_power = sum(s.propagate(0).power() / 2 for s in result[polarization])
        assert abs(ensemble_power - 1) < 1e-10

    # --- TM-like and TE-like sources carry identical angular power --------------
    for tm, te in zip(result["TM"], result["TE"]):
        np.testing.assert_allclose(
            np.sum(abs(tm.expansion.amplitudes) ** 2, axis=-1),
            np.sum(abs(te.expansion.amplitudes) ** 2, axis=-1),
            rtol=1e-10,
            atol=1e-10,
        )

    return result


# Dense-mask sources used throughout the notebook
sources = prepare_sources()

## Propagation pipeline

The three dense-mask cases use the same geometry and numerical settings. Each
case propagates its two coherent illumination fields through the interface;
their intensities are then averaged.

The helpers below solve, evaluate, form the image observable, measure a
registered dip and draw an intensity map.


In [ ]:
# --- Solving and evaluating ---------------------------------------------------
def solve(incident, mode, nodes=18):
    """Pass each coherent source through the interface; return solutions and time."""
    pipeline = optics.PropagationPipeline(
        optics.OpticalSystem([interface]),
        max_sine=0.90,
        transmission=optics.SpectralFresnel(
            coefficients=mode,
            nodes=nodes,
            angular_limit=0.60,
            cutoff=1e-8,
            cache_bytes=128 * 1024**2,
        ),
    )

    start = perf_counter()
    result = [pipeline.solve(source) for source in incident]
    return result, perf_counter() - start


def evaluate(solutions, grid):
    """Outgoing fields of every coherent source on one grid."""
    return [solution.evaluate(grid) for solution in solutions]


def image_plane(x, y):
    """Cartesian grid in the image plane z = F."""
    return optics.CartesianGrid(x, y, optics.Frame([0, 0, image_z]))


# --- Intensity observables ----------------------------------------------------
def intensity(fields):
    """Total electric intensity of an incoherent ensemble."""
    return sum(field.electric.norm2() / 2 for field in fields)


def image_intensity(fields, name):
    """Transverse intensity for the unity reference; total for physical fields."""
    if name == "Reference":
        return sum(
            np.sum(abs(field.electric.values[..., :2]) ** 2, axis=-1) / 2
            for field in fields
        )

    return intensity(fields)


# --- Registered dip between two bright positions -------------------------------
def dip(axis, profile, left, right, center):
    """D = 1 − I(center) / min[I(left), I(right)], all at nominal positions."""
    bright_left = np.interp(left, axis, profile)
    bright_right = np.interp(right, axis, profile)
    dark = np.interp(center, axis, profile)

    return 1 - dark / np.minimum(bright_left, bright_right)


# --- Intensity map on a linear 0–1 scale --------------------------------------
def intensity_map(ax, x, y, values):
    return ax.pcolormesh(
        x / um, y / um, values, shading="auto", cmap="hot", vmin=0, vmax=1
    )

## Dense-mask intensity images

These are plain intensity maps: $I_\perp$ for the unity reference, and $I_E$ for each physical Fresnel field. Every image uses the same intensity scale and equal physical x/y scales. **A–A** locates the longitudinal section, **B–B** the contact profile, and the rectangle the separate polarization close-up. The polarization-prepared result is an additional illumination experiment.

In [ ]:
# --- Focal-plane window -------------------------------------------------------
image_x = np.linspace(-5, 5, 301) * wavelength
image_y = np.linspace(-3, 3, 241) * wavelength
image_grid = image_plane(image_x, image_y)


# --- Solve each case and evaluate its focal image ------------------------------
solutions, fields, images = {}, {}, {}

for name, (polarization, mode) in cases.items():
    solutions[name], solve_seconds = solve(sources[polarization], mode)

    start = perf_counter()
    fields[name] = evaluate(solutions[name], image_grid)
    evaluation_seconds = perf_counter() - start

    images[name] = image_intensity(fields[name], name)

    # Modal power dropped by the transmission cutoff
    omitted = max(s.interactions[0].report.omitted_modal_l2 for s in solutions[name])

    print(
        f"{name}: solve {solve_seconds:.2f} s; "
        f"focal evaluation {evaluation_seconds:.2f} s; "
        f"omitted modal L2 {omitted:.2e}",
        flush=True,
    )

The three focal images on one shared intensity scale, with the cut lines and the close-up box marked:

In [ ]:
def mark_cuts(ax):
    """Draw section A–A, profile B–B and the edge close-up box."""

    # Section A–A: dashed vertical line
    ax.axvline(cut_x / um, color="cyan", ls="--", lw=0.6)
    ax.text(cut_x / um, image_y[-1] / um, "A–A", color="cyan", ha="center", va="bottom")

    # Profile B–B: dotted horizontal line across the contact pair
    contact_ends = np.array([1, 5]) * pitch_x / um
    ax.plot(contact_ends, [contact_y / um] * 2, color="lime", ls=":", lw=0.6)
    ax.text(
        contact_ends[1],
        contact_y / um,
        "B–B",
        color="lime",
        ha="right",
        va="bottom",
        fontsize=7,
    )

    # Edge close-up: outlined box
    corner = (edge_xlimits[0] / um, edge_ylimits[0] / um)
    ax.add_patch(
        Rectangle(
            corner,
            np.ptp(edge_xlimits) / um,
            np.ptp(edge_ylimits) / um,
            fill=False,
            edgecolor="cyan",
            lw=0.6,
        )
    )
    ax.text(
        corner[0],
        edge_ylimits[1] / um,
        "edge view",
        color="cyan",
        fontsize=7,
        va="bottom",
    )


# --- One figure per case, all on the same intensity scale -----------------------
common_peak = max(values.max() for values in images.values())

for name in cases:
    fig, ax = plt.subplots(figsize=(6, 3.9), layout="constrained")

    im = intensity_map(ax, image_x, image_y, images[name] / common_peak)
    mark_cuts(ax)

    ax.set(
        xlabel="Image x (µm)",
        ylabel="Image y (µm)",
        aspect="equal",
        title=labels[name],
    )
    fig.colorbar(im, ax=ax, label="Intensity / shared peak", fraction=0.035, pad=0.025)
    plt.show()

### Polarization at a comb edge

The box coordinates below select the **right ends of the lower-left comb tracks**. These two Fresnel views sample the same computed focal fields on a finer observation grid. Glyph size follows transverse intensity on a logarithmic display scale, with a 25% visibility floor; opacity still represents degree of polarization. The 0.2% intensity threshold includes weak edges. Ellipses show transverse polarization, while the background includes $E_z$.

In [ ]:
# --- Finer grid inside the edge box -------------------------------------------
edge_x = np.linspace(*edge_xlimits, 181)
edge_y = np.linspace(*edge_ylimits, 201)
edge_grid = image_plane(edge_x, edge_y)


for name in ("Fresnel", "Prepared illumination"):
    edge_fields = evaluate(solutions[name], edge_grid)

    fig, ax = plt.subplots(figsize=(3.6, 4), layout="constrained")

    # Background: total electric intensity on the shared focal scale
    im = intensity_map(ax, edge_x, edge_y, intensity(edge_fields) / common_peak)

    # Overlay: transverse polarization ellipses of the incoherent ensemble
    plot_polarization(
        edge_fields,
        weights=[0.5, 0.5],
        axis=ax,
        background=False,
        colorbar=False,
        crop=False,
        density=3,
        max_glyphs=1600,
        size_scale="log",
        size_floor=0.25,
        min_intensity_fraction=0.002,
        title=f"{name} · comb edge",
    )

    ax.set(xlabel="Image x (µm)", ylabel="Image y (µm)", aspect="equal")
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03, label="I / shared focal peak")
    plt.show()

## Occupied NA, throughput and registered resolution

Outgoing modal z-directed power is proportional to $k_z|\mathbf a|^2$. Report its fraction above **NA = 1** and the NA enclosing 95% of that power. This distinguishes a high nominal aperture from an actually high-angle field.

In [ ]:
for name in cases:
    # --- Outgoing modal power k_z |a|² against the NA of each mode ---------------
    angular_na, angular_power = [], []

    for solution in solutions[name]:
        expansion = solution.spectrum.expansion
        transverse_k = np.linalg.norm(expansion.wavevectors[:, :2].real, axis=-1)
        modal_power = expansion.wavevectors[:, 2].real * np.sum(
            abs(expansion.amplitudes) ** 2, axis=-1
        )

        angular_na.extend(transverse_k / k0)
        angular_power.extend(modal_power)

    angular_na = np.array(angular_na)
    angular_power = np.array(angular_power)

    # --- NA enclosing 95% of the power, and the fraction above NA = 1 -----------
    order = np.argsort(angular_na)
    cumulative = np.cumsum(angular_power[order])
    na95 = angular_na[order][np.searchsorted(cumulative, 0.95 * angular_power.sum())]

    high_fraction = angular_power[angular_na > 1].sum() / angular_power.sum()

    # --- Power through the focal window and its longitudinal share --------------
    detector_power = sum(field.power() / 2 for field in fields[name])

    longitudinal = sum(abs(field.electric.Ez) ** 2 / 2 for field in fields[name])
    ez_fraction = np.sum(longitudinal * image_grid.weights) / np.sum(
        intensity(fields[name]) * image_grid.weights
    )

    print(
        f"{name}: P(NA > 1) {high_fraction:.2%}; NA95 {na95:.3f}; "
        f"detector {detector_power:.4f} W; Ez {ez_fraction:.2%}"
    )

### Registered resolution

Resolution is measured at the nominal mask coordinates, without shifting peaks or gaps. A y cut crosses three fine gaps in the left comb; an x cut crosses the contact gap. Registered gap depth is $D=1-I_g/\min(I_\ell,I_r)$; negative values are retained. Curves use their own peak to expose shape, while focal maps above retain a shared absolute scale. Detector power covers only the displayed window. Unity-case flux is a formal control, not a dielectric efficiency.

All gap depths use the same intensity observable as their corresponding image: transverse for the reference, total electric intensity for the physical fields. The reported longitudinal fraction always uses total electric intensity as its denominator.

In [ ]:
# --- A–A through the lower comb: four tracks and the three gaps between them ----
line_y = np.linspace(-9, -1, 501) * pitch_y
line_points = np.column_stack(
    (np.full_like(line_y, cut_x), line_y, np.full_like(line_y, image_z))
)
line_grid = optics.PointGrid(line_points)

track_y = np.arange(-8, -1, 2) * pitch_y
gap_y = (track_y[:-1] + track_y[1:]) / 2


# --- B–B across the contact pair ----------------------------------------------
contact_x = np.linspace(1, 5, 301) * pitch_x
contact_points = np.column_stack(
    (contact_x, np.full_like(contact_x, contact_y), np.full_like(contact_x, image_z))
)
contact_grid = optics.PointGrid(contact_points)


def gap_depth(profile):
    """Registered depth of each comb gap on A–A."""
    return dip(line_y, profile, track_y[:-1], track_y[1:], gap_y)


# --- Comb gap depths along y --------------------------------------------------
profiles = {}

for name in cases:
    profiles[name] = image_intensity(evaluate(solutions[name], line_grid), name)

    depths = np.round(100 * gap_depth(profiles[name]), 2)
    print(f"{name}: registered y gap depths (%):", depths)


# --- Contact gap depth along x ------------------------------------------------
contact_profiles = {}

for name in cases:
    contact_fields = evaluate(solutions[name], contact_grid)
    contact_profiles[name] = image_intensity(contact_fields, name)

    depth = dip(
        contact_x, contact_profiles[name], 2 * pitch_x, 4 * pitch_x, 3 * pitch_x
    )
    print(f"{name}: registered x contact-gap depth {depth:.2%}")

Both profiles, each normalized to its own peak; dotted lines mark the registered gap positions.

In [ ]:
# (direction, axis, profiles, gap positions, title)
views = (
    ("y", line_y, profiles, gap_y, "Lower comb on A–A"),
    ("x", contact_x, contact_profiles, [3 * pitch_x], "B–B: contact separation"),
)

for direction, axis, curves, gaps, title in views:
    fig, ax = plt.subplots(figsize=(6, 2.6), layout="constrained")

    # One curve per case
    for name in cases:
        ax.plot(axis / um, curves[name] / curves[name].max(), label=labels[name])

    # Registered gap positions
    for gap in gaps:
        ax.axvline(gap / um, color="grey", ls=":", lw=0.6)

    ax.set(xlabel=f"Image {direction} (µm)", ylabel="Intensity / own peak", title=title)
    ax.legend()
    plt.show()

How much does each physical image differ in shape from the reference? Each image is first normalized to unit integral over the focal window.

In [ ]:
# --- Normalize each image to unit integral over the focal window ----------------
normalized = {
    name: image / np.sum(image * image_grid.weights) for name, image in images.items()
}


# --- Relative L2 difference from the reference --------------------------------
reference = normalized["Reference"]

for name in ("Fresnel", "Prepared illumination"):
    change = np.linalg.norm(normalized[name] - reference) / np.linalg.norm(reference)
    print(f"Integral-normalized image change, {name} versus Reference: {change:.2%}")

### Why the intensity images differ

For symmetric directions $\mathbf d_\pm=(0,\pm\sin\theta,\cos\theta)$, the p-polarization vectors are $\mathbf e_{p,\pm}=(0,\cos\theta,\mp\sin\theta)$. With equal amplitudes and $q=k\sin\theta$,

$$I_\perp(y)\propto\cos^2\theta\cos^2(qy),\qquad
I_E(y)\propto\cos^2\theta\cos^2(qy)+\sin^2\theta\sin^2(qy).$$

The transverse reference has the scalar fringe shape after normalization. At $\theta=45^\circ$, the longitudinal contribution fills its dark fringes and the total electric intensity becomes uniform. For s polarization, both electric vectors point along x, retaining full fringe modulation.

The finite-mask pipeline includes many directions, aberrations and Fresnel weighting. Its measured profiles, rather than the two-wave approximation, determine the actual resolution differences. The reference-to-Fresnel comparison changes both the transmission coefficients and the intensity observable; the prepared-illumination comparison uses total electric intensity in both physical cases.

## Longitudinal section A–A

Follow the dashed **A–A** line on the focal images: this is the yz section at fixed $x=-1.20$ µm. The lower half crosses the comb; the upper half crosses the meander. It shows how those particular features evolve through the image plane, rather than a view of the entire mask.

The narrow strip beside each field is the **nominal mask slice on A–A**, with exactly the same y scale. The cyan line at $z-F=0$ is the focal plane. All three field maps share one intensity scale and equal physical y/z scales.

As in the focal images, the reference shows $I_\perp$ and both physical fields show $I_E$.

In [ ]:
# --- yz section at x = cut_x, ±3 λ around the image plane -----------------------
section_y = np.linspace(-3, 3, 181) * wavelength
section_z = image_z + np.linspace(-3, 3, 181) * wavelength

Y, Z = np.meshgrid(section_y, section_z)
section_points = np.stack((np.full_like(Y, cut_x), Y, Z), axis=-1)
section_grid = optics.PointGrid(section_points)


# --- Intensity of each case on the section, with one shared peak ----------------
sections = {
    name: image_intensity(evaluate(solutions[name], section_grid), name)
    for name in cases
}
section_peak = max(values.max() for values in sections.values())

# Mask column crossed by A–A
mask_column = int(np.floor(cut_x / pitch_x + mask.shape[1] / 2))


for name in cases:
    fig, ax = plt.subplots(figsize=(4.8, 4), layout="constrained")

    # Field map, z horizontal and y vertical
    defocus = section_z - image_z
    im = intensity_map(ax, defocus, section_y, sections[name].T / section_peak)

    # Image plane
    ax.axvline(0, color="cyan", ls=":", lw=0.8)
    ax.text(
        0.04,
        0.98,
        "image plane",
        transform=ax.get_xaxis_transform(),
        color="cyan",
        va="top",
        fontsize=7,
        rotation=90,
    )

    ax.set(xlabel="z − F (µm)", aspect="equal", title=f"{labels[name]} · A–A")
    ax.tick_params(labelleft=False)

    # Narrow strip on the left: the nominal mask slice on A–A, same y axis
    divider = make_axes_locatable(ax)
    strip = divider.append_axes("left", size="10%", pad=0.12, sharey=ax)

    strip.pcolormesh(
        [0, 1], y_edges, mask[:, mask_column, None], cmap="gray", vmin=0, vmax=1
    )
    strip.set(xticks=[], ylabel="y (µm)", ylim=(section_y[0] / um, section_y[-1] / um))
    strip.set_facecolor("black")
    strip.set_title("Mask\nA–A", fontsize=8)

    # Color scale on the right
    color_axis = divider.append_axes("right", size="4%", pad=0.10)
    fig.colorbar(im, cax=color_axis, label="I / shared section peak")
    plt.show()

Interpret resolution from the intensity images and their registered profiles together. Separated lobes or deep minima alone do not guarantee faithful positions and shapes. All images use the public pipeline with the specified geometry, bandwidth and finite mask.

For the three registered fine-comb gaps, the transverse unity reference gives depths of **80–94%**, the physical Fresnel field **16–20%**, and the polarization-prepared field **79–94%**. These values refer to the intensity images shown above, using the same nominal measurement positions.

The fields use the forward tangent-plane model with the specified finite graph continuation and virtual-mask illumination. These images do not establish full-wave accuracy for every lens or replace the independent comparisons in notebook 06.
